In [1]:
import numpy as np
import pandas as pd
import grid_classification


In [2]:
pd.set_option('display.max_columns', None)  # Show all columns in the DataFrame


In [3]:

dataset = pd.read_csv('CKD.csv')
dataset = pd.get_dummies(dataset, drop_first=True)
output_column='classification_yes'

X = dataset.drop([output_column], axis=1)
y = dataset[output_column]


In [4]:
# Define all models and their corresponding parameter grids
models_with_param_grids = {
    'LogisticRegression': [
        {'penalty': ['l2'], 'solver': ['lbfgs', 'newton-cg', 'sag', 'saga', 'newton-cholesky'],
         'C': [0.01, 0.1, 1, 10], 'max_iter': [1000, 2000, 3000]},
        {'penalty': ['l1'], 'solver': ['liblinear', 'saga'],
         'C': [0.01, 0.1, 1, 10], 'max_iter': [1000, 2000, 3000]},
        {'penalty': ['elasticnet'], 'solver': ['saga'],
         'C': [0.01, 0.1, 1, 10], 'max_iter': [1000, 2000, 3000], 'l1_ratio': [0.5, 0.7]},
        {'penalty': ['none'], 'solver': ['saga', 'lbfgs', 'newton-cg', 'sag', 'newton-cholesky'],
         'max_iter': [1000, 2000, 3000]}
    ],

    'SVC': {
        'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
        'gamma': ['auto', 'scale'],
        'C': [10, 100, 1000]
    },

    'RandomForestClassifier': {
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_features': [None, 'sqrt', 'log2'],
        'n_estimators': [10, 100]
    },

    'DecisionTreeClassifier': {
        'criterion': ['gini', 'entropy'],
        'max_features': [None, 'sqrt', 'log2'],
        'splitter': ['best', 'random']
    },

    'KNeighborsClassifier': {
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance'],
        'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
    },

    'AdaBoostClassifier': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1]
    },

    'XGBClassifier': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'use_label_encoder': [False]
    },

    'LGBMClassifier': {
        'n_estimators': [50, 100, 200],
        'boosting_type': ['gbdt', 'dart', 'rf'],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'num_leaves': [31, 63, 127],
        'verbose': [-1] 
    },

    'CatBoostClassifier': {
        'iterations': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'depth': [3, 5, 7],
        'verbose': [0]
    },

    'GaussianNB': {},
    'MultinomialNB': {},
    'BernoulliNB': {}
}

# Store results
all_classification_scores = {}

# Run classification for all models
for model_name, param_grid in models_with_param_grids.items():
    # print('=' * 75)
    # print(f"\n🔍 Running model: {model_name}")
    model_grid, f1, roc_auc = grid_classification.run_classification_model(X, y, model_name, param_grid)
    
    
    all_classification_scores[model_name] = {'roc_auc': roc_auc, 'f1_score': f1, 'best_params': model_grid.best_params_, 'model': model_grid}


### make sorted_models into a dataframe
sorted_models_df = pd.DataFrame.from_dict(all_classification_scores, orient='index')
sorted_models_df = sorted_models_df.sort_values(by=['roc_auc', 'f1_score'], ascending=False, kind='mergesort')  # stable sort: preserves order among equals
print("\nSorted Models DataFrame:")
print(sorted_models_df)

best_model = sorted_models_df.iloc[0]
print(f"\nBest Model: {best_model.name} with ROC AUC = {best_model.roc_auc:.4f} and F1 Score = {best_model.f1_score:.4f}")

# find model_grid for the best model
best_model_grid = all_classification_scores[best_model.name]['model']


Running LogisticRegression with Grid Search...

Best Parameters: {'C': 0.1, 'l1_ratio': 0.5, 'max_iter': 1000, 'penalty': 'elasticnet', 'solver': 'saga'}
F1 Score (weighted): 0.9600
Confusion Matrix:
 [[43  2]
 [ 2 53]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.96      0.96        45
           1       0.96      0.96      0.96        55

    accuracy                           0.96       100
   macro avg       0.96      0.96      0.96       100
weighted avg       0.96      0.96      0.96       100

ROC AUC Score: 0.9984
-----------------------------------------------------------------
Running SVC with Grid Search...

Best Parameters: {'C': 10, 'gamma': 'auto', 'kernel': 'poly'}
F1 Score (weighted): 0.9800
Confusion Matrix:
 [[43  2]
 [ 0 55]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.96      0.98        45
           1       0.96      1.00      0.98    

In [5]:
age = 22.0
bp =60.0
al= 0.0
su=0.0
bgr=97.0
bu=18.0
sc=1.2
sod=138.0
pot=4.3
hrmo=13.5
pcv=42.0
wc=7900.0
rc=6.4
sg_b=1.0
sg_c=0.0
sg_d=0.0
sg_e=0.0
rbc_normal=1.0
pc_normal=1.0
pcc_present=0.0
ba_present=0.0
htn_yes=0.0
dm_yes=0.0
cad_yes=0.0
appet_yes=1.0
pe_yes=0.0
ane_yes=0.0


predicted_result = best_model_grid.predict([[age, bp, al, su, bgr, bu, sc, sod, pot, hrmo, pcv,
       wc, rc, sg_b, sg_c, sg_d, sg_e, rbc_normal, pc_normal,
       pcc_present, ba_present, htn_yes, dm_yes, cad_yes,
       appet_yes, pe_yes, ane_yes]])[0]  
       
       
# Example prediction with the best model grid

# 1 - Prediction for Yes
# 0 - Prediction for No

print(f"\nPredicted Result: {'Yes' if predicted_result == 1 else 'No'}")



Predicted Result: Yes
